# WikiHow-MY — NLLB-200-distilled-600M fine-tune (en→my) on Colab

Heavy GPU training only. Everything else (data, eval, paper) runs locally via Claude Code.

**Before running:** upload the project folder (with `data/processed/`, `src/`) to your Google Drive, e.g. `MyDrive/wikihow-my/`. Set `PROJECT` below. Runtime → Change runtime type → **GPU (A100 preferred, T4 ok)**.

In [ ]:
!pip -q install -U transformers datasets sacrebleu unbabel-comet sentencepiece pyyaml accelerate myanmar-tools

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/wikihow-my'  # <-- edit to your path
os.chdir(PROJECT)
print('cwd:', os.getcwd())
assert os.path.exists('data/processed/train.jsonl'), 'upload data/processed/ first'

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Fine-tune (early-stops on dev chrF)

In [ ]:
!python src/train/finetune_nllb.py --config src/train/config.yaml --out_dir /content/ckpt
# On T4 OOM: lower per_device_train_batch_size to 4 and gradient_accumulation_steps to 16 in config.yaml

## 2. Inference: zero-shot + fine-tuned on the test set

In [ ]:
!python src/infer/translate.py --split test --system nllb_zeroshot
!python src/infer/translate.py --split test --system nllb_finetuned --model /content/ckpt/best

## 3. Score (chrF++ / spBLEU / BLEU / COMET)

In [ ]:
# COMET runs here (GPU). Edit automatic.py's COMET hook to load 'Unbabel/wmt22-comet-da' if desired.
!python src/eval/automatic.py --hyps experiments/results/nllb_zeroshot_test_hyps.txt --refs data/processed/test.jsonl --system nllb_zeroshot
!python src/eval/automatic.py --hyps experiments/results/nllb_finetuned_test_hyps.txt --refs data/processed/test.jsonl --system nllb_finetuned
!python src/eval/make_tables.py

## 4. Persist results back to Drive
`experiments/results/*.json`, `*_hyps.txt`, and `paper/tables/main_results.tex` are already under `PROJECT` (Drive). Also copy `/content/ckpt/best` to Drive if you want to keep the checkpoint.

**Exit criterion (Week 2):** fine-tuned chrF++ on dev beats zero-shot. If not → check data leakage / lower LR / more epochs (see plan §13).